<a href="https://colab.research.google.com/github/amasha-sinhalagoda/J26-IT-334/blob/Landslide-PreProcess-Dataset/Landslidepreprocessnew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!apt-get install unrar -y -qq
!unrar x "NuwaraEliya(SLIIT)-(Bagya).rar" -o+


UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from NuwaraEliya(SLIIT)-(Bagya).rar

Extracting  DATA_SHARING_GUIDE.pdf                                         0%  OK 
Creating    Dithawa_HR_By_District                                    OK
Creating    Dithawa_HR_By_District/Nuwara_Eliya                       OK
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.cpg       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.dbf       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.prj       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.shp       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.shx       0%  OK 
Creating    LHZM                                                      OK
Creating    LHZM/

In [3]:
!pip install geopandas pandas openpyxl xlsxwriter shapely -q

import geopandas as gpd
import pandas as pd
import numpy as np
import os
from google.colab import files

print("Dependencies loaded successfully!")

# Locate shapefiles
inspections_shp = None
lhmp_shp = None

for root, dirs, files_list in os.walk("."):
    for file in files_list:
        if "Dithawa" in file and file.endswith(".shp"):
            inspections_shp = os.path.join(root, file)
        elif "LHMP" in file and file.endswith(".shp"):
            lhmp_shp = os.path.join(root, file)

print(f"Inspections shapefile found: {inspections_shp}")
print(f"LHMP shapefile found: {lhmp_shp}")

gdf_inspections = gpd.read_file(inspections_shp).to_crs(epsg=5235)
gdf_lhmp = gpd.read_file(lhmp_shp).to_crs(epsg=5235)

print(f"Loaded {len(gdf_inspections)} inspection points.")
print(f"Loaded {len(gdf_lhmp)} LHMP hazard polygons.")

# Keep an untouched copy for the "Raw_Attributes" sheet
df_raw = pd.DataFrame(gdf_inspections.drop(columns="geometry")).copy()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 2.6 MB/s eta 0:00:00
Dependencies loaded successfully!
Inspections shapefile found: ./Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.shp
LHMP shapefile found: ./LHZM/Nuwara_Eliya/Nuwara_Eliya_LHMP_50000.shp
Loaded 507 inspection points.
Loaded 3083 LHMP hazard polygons.


In [5]:
# Spatial coordinates
gdf_inspections["X_Coord"] = gdf_inspections.geometry.x
gdf_inspections["Y_Coord"] = gdf_inspections.geometry.y

# Positive class label — these are all confirmed inspection points
gdf_inspections["Landslide_Occurrence"] = 1

# Encode HR1/HR2/HR3 risk codes into a severity rank
def parse_hr_level(val):
    if pd.isna(val): return 0
    val_str = str(val).upper()
    if "HR1" in val_str: return 3
    if "HR2" in val_str: return 2
    if "HR3" in val_str: return 1
    return 0

risk_cols = [c for c in gdf_inspections.columns if "RISK" in c.upper() or "HR" in c.upper()]
if risk_cols:
    gdf_inspections["Severity_Rank"] = gdf_inspections[risk_cols[0]].apply(parse_hr_level)
else:
    gdf_inspections["Severity_Rank"] = 1

# --- FIX: the LHMP layer's hazard field is literally named "Range" (values 1.0-4.0,
# plus a 100.0 no-data code), not anything containing "HAZARD" or "ZONE". The original
# substring match never found it, so every row silently became "Unmapped". ---
joined_gdf = gpd.sjoin(gdf_inspections, gdf_lhmp, how="left", predicate="intersects")

hazard_cols = [c for c in joined_gdf.columns
               if c.strip().lower() in ("range", "hazard", "hazard_zone", "zone")
               or "hazard" in c.lower() or "zone" in c.lower()]

if hazard_cols:
    # Group by the original index of gdf_inspections (which is the index of joined_gdf)
    # and take the first hazard zone if multiple are found for a single inspection point.
    # Then reindex to align with gdf_inspections.index, filling NaNs (for points with no intersection)
    # and then filling "Unmapped" for all NaNs.
    hazard_data = joined_gdf.groupby(joined_gdf.index)[hazard_cols[0]].first()
    gdf_inspections["NBRO_Hazard_Zone"] = hazard_data.reindex(gdf_inspections.index).fillna("Unmapped")
else:
    gdf_inspections["NBRO_Hazard_Zone"] = "Unmapped"

matched = (gdf_inspections["NBRO_Hazard_Zone"] != "Unmapped").sum()
print(f"Hazard zone matched for {matched} / {len(gdf_inspections)} points "
      f"({matched/len(gdf_inspections):.1%})")

# Flag the invalid Range=100.0 no-data code so it doesn't get treated as a real hazard level
gdf_inspections["Hazard_Zone_Flag"] = np.where(
    gdf_inspections["NBRO_Hazard_Zone"] == 100.0, "Invalid/No-data", "Valid"
)

df_ml_matrix = pd.DataFrame(gdf_inspections.drop(columns="geometry"))

Hazard zone matched for 506 / 507 points (99.8%)


In [7]:
# All rows so far are positive (Landslide_Occurrence = 1). A classifier needs negatives too.
# Generate random background points within the LHMP layer's bounding box that DON'T
# fall on a real inspection point, and mark them as negative class.
rng = np.random.default_rng(42)
minx, miny, maxx, maxy = gdf_lhmp.total_bounds

n_negatives = len(gdf_inspections)  # 1:1 balance — adjust as needed
neg_x = rng.uniform(minx, maxx, size=n_negatives * 3)  # oversample, then filter
neg_y = rng.uniform(miny, maxy, size=n_negatives * 3)

from shapely.geometry import Point
neg_points = gpd.GeoDataFrame(
    geometry=[Point(x, y) for x, y in zip(neg_x, neg_y)],
    crs=gdf_lhmp.crs
)

# Drop candidate points that are within ~200m of a real inspection point
buffer = gdf_inspections.geometry.buffer(200).union_all()
neg_points = neg_points[~neg_points.geometry.within(buffer)].head(n_negatives).reset_index(drop=True)

neg_points["X_Coord"] = neg_points.geometry.x
neg_points["Y_Coord"] = neg_points.geometry.y
neg_points["Landslide_Occurrence"] = 0
neg_points["Severity_Rank"] = 0

neg_joined = gpd.sjoin(neg_points, gdf_lhmp, how="left", predicate="intersects")
if hazard_cols:
    # Group by the original index of neg_points (which is the index of neg_joined)
    # and take the first hazard zone if multiple are found for a single negative point.
    # Then reindex to align with neg_points.index, filling NaNs (for points with no intersection)
    # and then filling "Unmapped" for all NaNs.
    neg_hazard_data = neg_joined.groupby(neg_joined.index)[hazard_cols[0]].first()
    neg_points["NBRO_Hazard_Zone"] = neg_hazard_data.reindex(neg_points.index).fillna("Unmapped")
else:
    neg_points["NBRO_Hazard_Zone"] = "Unmapped"
neg_points["Hazard_Zone_Flag"] = np.where(
    neg_points["NBRO_Hazard_Zone"] == 100.0, "Invalid/No-data", "Valid"
)

df_negatives = pd.DataFrame(neg_points.drop(columns="geometry"))

# Combine positive + negative samples for the final ML matrix
df_ml_matrix_balanced = pd.concat([df_ml_matrix, df_negatives], ignore_index=True)
print(f"Final matrix: {(df_ml_matrix_balanced['Landslide_Occurrence']==1).sum()} positive, "
      f"{(df_ml_matrix_balanced['Landslide_Occurrence']==0).sum()} negative")

Final matrix: 507 positive, 507 negative


In [8]:
excel_filename = "Landslide_Dataset_Preprocessed.xlsx"

with pd.ExcelWriter(excel_filename, engine="xlsxwriter") as writer:
    df_ml_matrix_balanced.to_excel(writer, sheet_name="ML_Feature_Matrix", index=False)
    df_raw.to_excel(writer, sheet_name="Raw_Attributes", index=False)  # genuinely untouched now

    summary_df = pd.DataFrame({
        "Attribute": [
            "Positive samples", "Negative samples", "Total records",
            "CRS Projection", "Total Columns", "Hazard match rate"
        ],
        "Value": [
            (df_ml_matrix_balanced["Landslide_Occurrence"]==1).sum(),
            (df_ml_matrix_balanced["Landslide_Occurrence"]==0).sum(),
            len(df_ml_matrix_balanced),
            "EPSG:5235 (Sri Lanka Grid)",
            len(df_ml_matrix_balanced.columns),
            f"{matched/len(gdf_inspections):.1%}"
        ]
    })
    summary_df.to_excel(writer, sheet_name="Metadata_Summary", index=False)

print(f"SUCCESS: '{excel_filename}' generated.")
files.download(excel_filename)

SUCCESS: 'Landslide_Dataset_Preprocessed.xlsx' generated.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>